# **XGBOOST TRAINING**

In [6]:
#!pip install xgboost

"""
TRAINING NOTEBOOK FOR IMBALANCED SUNT DATASET - XGBOOST
========================================================
"""

import pandas as pd
import numpy as np
import pickle
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from xgboost import XGBClassifier
from google.colab import drive
drive.mount('/content/drive')

print("✅ Libraries imported")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Libraries imported


In [7]:
# ============================================
# CELL 1: LOAD DATA
# ============================================

DRIVE_PATH = '/content/drive/MyDrive/Occupancy_capstone/Dataset/'
X_PATH = DRIVE_PATH + 'sunt_X_2024_03_march.parquet'
Y_PATH = DRIVE_PATH + 'sunt_y_2024_03_march.pkl'

X = pd.read_parquet(X_PATH)
with open(Y_PATH, 'rb') as f:
    y = pickle.load(f)

print("\n" + "=" * 70)
print("📊 DATASET SUMMARY")
print("=" * 70)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts(normalize=True).sort_index())


📊 DATASET SUMMARY
X shape: (8544587, 14)
y shape: (8544587,)

Target distribution:
occupancy_level
high         0.116677
low          0.577081
medium       0.272316
very_high    0.033927
Name: proportion, dtype: float64


In [8]:
# ============================================
# CELL 2: ANALYZE CLASS IMBALANCE
# ============================================

print("\n" + "=" * 70)
print("⚠️  CLASS IMBALANCE ANALYSIS")
print("=" * 70)

class_counts = y.value_counts()
majority_class = class_counts.max()
minority_class = class_counts.min()
imbalance_ratio = majority_class / minority_class

print(f"\nClass counts:")
for cls, count in class_counts.sort_index().items():
    pct = count / len(y) * 100
    bar = "█" * int(pct / 2)
    print(f"   {cls:10s}: {count:>10,} ({pct:5.2f}%) {bar}")

print(f"\nImbalance ratio: {imbalance_ratio:.1f}:1")
print(f"Majority class: {class_counts.idxmax()} ({class_counts.max():,})")
print(f"Minority class: {class_counts.idxmin()} ({class_counts.min():,})")

# ============================================
# OPCIÓN A: NO BALANCEAR + CLASS WEIGHTS (RECOMENDADO)
# ============================================

print("\n" + "=" * 70)
print("⚖️  CLASS BALANCE STRATEGY: WEIGHTED TRAINING")
print("=" * 70)

print("\n📊 Current class distribution:")
print("-" * 70)
dist = y.value_counts(normalize=True).sort_index()
for cls, prop in dist.items():
    count = (y == cls).sum()
    bar = "█" * int(prop * 50)
    print(f"   {cls:>10s}: {count:>10,} ({prop*100:>5.1f}%) {bar}")

# Calculate class weights
classes = np.unique(y)
class_weights_array = compute_class_weight(
    'balanced',
    classes=classes,
    y=y
)
class_weights = dict(zip(classes, class_weights_array))

print("\n📊 Computed class weights:")
print("-" * 70)
for cls, weight in sorted(class_weights.items()):
    print(f"   {cls:>10s}: {weight:>6.3f}")

print("\n💡 Interpretation:")
print("   • Higher weight = Model will pay more attention to this class")
print("   • Low weight = Majority class (less important)")
print("   • This balances without throwing away data!")

print(f"\n✅ Strategy: Use ALL {len(X):,} records with class weights")
print("   (No data is discarded)")
print("=" * 70)


⚠️  CLASS IMBALANCE ANALYSIS

Class counts:
   high      :    996,957 (11.67%) █████
   low       :  4,930,915 (57.71%) ████████████████████████████
   medium    :  2,326,825 (27.23%) █████████████
   very_high :    289,890 ( 3.39%) █

Imbalance ratio: 17.0:1
Majority class: low (4,930,915)
Minority class: very_high (289,890)

⚖️  CLASS BALANCE STRATEGY: WEIGHTED TRAINING

📊 Current class distribution:
----------------------------------------------------------------------
         high:    996,957 ( 11.7%) █████
          low:  4,930,915 ( 57.7%) ████████████████████████████
       medium:  2,326,825 ( 27.2%) █████████████
    very_high:    289,890 (  3.4%) █

📊 Computed class weights:
----------------------------------------------------------------------
         high:  2.143
          low:  0.433
       medium:  0.918
    very_high:  7.369

💡 Interpretation:
   • Higher weight = Model will pay more attention to this class
   • Low weight = Majority class (less important)
   • This b

In [9]:
# ============================================
# CELL 3: TRAIN/TEST SPLIT (STRATIFIED)
# ============================================

print("\n" + "=" * 70)
print("🔢 ENCODING TARGET FOR XGBOOST")
print("=" * 70)

from sklearn.preprocessing import LabelEncoder

# XGBoost requires numeric labels (0, 1, 2, 3)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Original classes: {label_encoder.classes_}")
print(f"Encoded as: {np.unique(y_encoded)}")
print(f"\nMapping:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"   {cls:>10s} → {i}")

print("\n" + "=" * 70)
print("✂️  TRAIN/TEST SPLIT (STRATIFIED)")
print("=" * 70)

X_train, X_test, y_train_encoded, y_test_encoded = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# Keep original labels for reporting
_, _, y_train_original, y_test_original = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTrain set: {len(X_train):,} records")
print(f"Test set:  {len(X_test):,} records")

print(f"\nTrain distribution:")
for cls, pct in y_train_original.value_counts(normalize=True).sort_index().items():
    print(f"   {cls}: {pct*100:.2f}%")

print(f"\nTest distribution:")
for cls, pct in y_test_original.value_counts(normalize=True).sort_index().items():
    print(f"   {cls}: {pct*100:.2f}%")


🔢 ENCODING TARGET FOR XGBOOST
Original classes: ['high' 'low' 'medium' 'very_high']
Encoded as: [0 1 2 3]

Mapping:
         high → 0
          low → 1
       medium → 2
    very_high → 3

✂️  TRAIN/TEST SPLIT (STRATIFIED)

Train set: 6,835,669 records
Test set:  1,708,918 records

Train distribution:
   high: 11.67%
   low: 57.71%
   medium: 27.23%
   very_high: 3.39%

Test distribution:
   high: 11.67%
   low: 57.71%
   medium: 27.23%
   very_high: 3.39%


In [10]:
# ============================================
# CELL 4: CALCULATE SAMPLE WEIGHTS
# ============================================

print("\n" + "=" * 70)
print("⚙️  CALCULATING SAMPLE WEIGHTS FOR XGBOOST")
print("=" * 70)

# XGBoost usa sample_weight (uno por cada muestra de entrenamiento)
# ⚠️ IMPORTANTE: Usar y_train_original (strings) para calcular weights
sample_weights_train = compute_sample_weight(class_weights, y_train_original)

print(f"Sample weights calculated for training set:")
print(f"   Total samples: {len(sample_weights_train):,}")
print(f"   Min weight: {sample_weights_train.min():.4f}")
print(f"   Max weight: {sample_weights_train.max():.4f}")
print(f"   Mean weight: {sample_weights_train.mean():.4f}")
print(f"   Weight ratio: {sample_weights_train.max() / sample_weights_train.min():.2f}:1")

print("\n💡 How it works:")
print("   • Each training sample gets a weight")
print("   • Minority class samples have HIGHER weights")
print("   • Model pays more attention to rare classes")



⚙️  CALCULATING SAMPLE WEIGHTS FOR XGBOOST
Sample weights calculated for training set:
   Total samples: 6,835,669
   Min weight: 0.4332
   Max weight: 7.3688
   Mean weight: 1.0000
   Weight ratio: 17.01:1

💡 How it works:
   • Each training sample gets a weight
   • Minority class samples have HIGHER weights
   • Model pays more attention to rare classes


In [11]:
# ============================================
# CELL 5: TRAINING XGBOOST MODEL
# ============================================

print("\n" + "=" * 70)
print("🤖 TRAINING XGBOOST (MULTICLASS WITH WEIGHTS)")
print("=" * 70)

# ⚠️ FIXED: Removed num_class parameter (not needed in XGBClassifier)
model = XGBClassifier(
    objective="multi:softprob",
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,  # ⭐ Bajado de 0.1 a 0.05 (más lento = mejor)
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,  # ⭐ Reducido de 1 (menos agresivo)
    reg_lambda=1,  # ⭐ Reducido de 2 (L2 regularization)
    min_child_weight=5,  # ⭐ AGREGADO (ayuda con imbalance)
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    verbosity=1  # ⭐ AGREGADO: Ver progreso
)

print(f"\n📊 XGBoost Configuration:")
print(f"   objective: {model.objective}")
print(f"   n_estimators: {model.n_estimators}")
print(f"   max_depth: {model.max_depth}")
print(f"   learning_rate: {model.learning_rate}")
print(f"   subsample: {model.subsample}")
print(f"   colsample_bytree: {model.colsample_bytree}")
print(f"   Using sample_weight: ✅")

print("\n⏳ Training XGBoost... (this may take ~30 minutes)")
start_time = time.time()

# ⭐ CRÍTICO: Usar y_train_encoded (números) y sample_weight
model.fit(
    X_train,
    y_train_encoded,  # ⚠️ CAMBIO: Usar versión encoded
    sample_weight=sample_weights_train,
    eval_set=[(X_train, y_train_encoded), (X_test, y_test_encoded)],  # ⚠️ CAMBIO
    verbose=50
)

elapsed = time.time() - start_time
print(f"\n✅ Training completed in {elapsed:.2f} seconds ({elapsed/60:.1f} minutes)")



🤖 TRAINING XGBOOST (MULTICLASS WITH WEIGHTS)

📊 XGBoost Configuration:
   objective: multi:softprob
   n_estimators: 300
   max_depth: 8
   learning_rate: 0.05
   subsample: 0.8
   colsample_bytree: 0.8
   Using sample_weight: ✅

⏳ Training XGBoost... (this may take 5-10 minutes)
[0]	validation_0-mlogloss:1.37399	validation_1-mlogloss:1.37396
[50]	validation_0-mlogloss:1.07694	validation_1-mlogloss:1.07693
[100]	validation_0-mlogloss:1.00215	validation_1-mlogloss:1.00269
[150]	validation_0-mlogloss:0.96151	validation_1-mlogloss:0.96246
[200]	validation_0-mlogloss:0.93227	validation_1-mlogloss:0.93367
[250]	validation_0-mlogloss:0.91017	validation_1-mlogloss:0.91193
[299]	validation_0-mlogloss:0.89347	validation_1-mlogloss:0.89557

✅ Training completed in 2046.62 seconds (34.1 minutes)


In [12]:
# ============================================
# CELL 6: EVALUATION WITH PROPER METRICS
# ============================================
print("\n" + "=" * 70)
print("📊 MODEL EVALUATION")
print("=" * 70)

# Predictions (en formato encoded)
y_pred_train_encoded = model.predict(X_train)
y_pred_test_encoded = model.predict(X_test)

# Decode back to original labels for reporting
y_pred_train = label_encoder.inverse_transform(y_pred_train_encoded)
y_pred_test = label_encoder.inverse_transform(y_pred_test_encoded)

# Standard accuracy
acc_train = accuracy_score(y_train_original, y_pred_train)
acc_test = accuracy_score(y_test_original, y_pred_test)

# ⭐ BALANCED ACCURACY - Better metric for imbalanced data
bal_acc_train = balanced_accuracy_score(y_train_original, y_pred_train)
bal_acc_test = balanced_accuracy_score(y_test_original, y_pred_test)

# F1 Score (weighted)
f1_train = f1_score(y_train_original, y_pred_train, average='weighted')
f1_test = f1_score(y_test_original, y_pred_test, average='weighted')

# F1 Score (macro - treats all classes equally)
f1_macro_train = f1_score(y_train_original, y_pred_train, average='macro')
f1_macro_test = f1_score(y_test_original, y_pred_test, average='macro')

print("\n" + "-" * 70)
print("METRICS COMPARISON:")
print("-" * 70)
print(f"{'Metric':<25} {'Train':>12} {'Test':>12} {'Gap':>12}")
print("-" * 70)
print(f"{'Accuracy':<25} {acc_train:>12.4f} {acc_test:>12.4f} {acc_train-acc_test:>12.4f}")
print(f"{'Balanced Accuracy ⭐':<25} {bal_acc_train:>12.4f} {bal_acc_test:>12.4f} {bal_acc_train-bal_acc_test:>12.4f}")
print(f"{'F1 (weighted)':<25} {f1_train:>12.4f} {f1_test:>12.4f} {f1_train-f1_test:>12.4f}")
print(f"{'F1 (macro) ⭐':<25} {f1_macro_train:>12.4f} {f1_macro_test:>12.4f} {f1_macro_train-f1_macro_test:>12.4f}")
print("-" * 70)

# Check overfitting
gap = bal_acc_train - bal_acc_test
if gap > 0.1:
    print(f"\n⚠️  High gap ({gap:.2%}) - possible overfitting")
elif gap > 0.05:
    print(f"\n⚠️  Moderate gap ({gap:.2%}) - acceptable")
else:
    print(f"\n✅ Low gap ({gap:.2%}) - model generalizes well")

print("\n💡 KEY METRICS FOR IMBALANCED DATA:")
print(f"   • Balanced Accuracy: {bal_acc_test:.4f}")
print(f"   • F1 Macro: {f1_macro_test:.4f}")
print("   (These are more meaningful than regular accuracy)")



📊 MODEL EVALUATION

----------------------------------------------------------------------
METRICS COMPARISON:
----------------------------------------------------------------------
Metric                           Train         Test          Gap
----------------------------------------------------------------------
Accuracy                        0.6189       0.6172       0.0017
Balanced Accuracy ⭐             0.5934       0.5894       0.0039
F1 (weighted)                   0.6355       0.6339       0.0016
F1 (macro) ⭐                    0.5062       0.5031       0.0030
----------------------------------------------------------------------

✅ Low gap (0.39%) - model generalizes well

💡 KEY METRICS FOR IMBALANCED DATA:
   • Balanced Accuracy: 0.5894
   • F1 Macro: 0.5031
   (These are more meaningful than regular accuracy)


In [13]:
# ============================================
# CELL 7: DETAILED CLASSIFICATION REPORT
# ============================================

print("\n" + "=" * 70)
print("📋 CLASSIFICATION REPORT (TEST SET)")
print("=" * 70)
print(classification_report(y_test_original, y_pred_test))


📋 CLASSIFICATION REPORT (TEST SET)
              precision    recall  f1-score   support

        high       0.34      0.39      0.36    199392
         low       0.85      0.72      0.78    986183
      medium       0.46      0.48      0.47    465365
   very_high       0.27      0.77      0.40     57978

    accuracy                           0.62   1708918
   macro avg       0.48      0.59      0.50   1708918
weighted avg       0.67      0.62      0.63   1708918



In [14]:
# ============================================
# CELL 8: CONFUSION MATRIX
# ============================================
print("\n" + "=" * 70)
print("📊 CONFUSION MATRIX (TEST SET)")
print("=" * 70)

cm = confusion_matrix(y_test_original, y_pred_test)
labels = sorted(y.unique())

print(f"\n{'':>12}", end="")
for label in labels:
    print(f"{label:>12}", end="")
print(" ← Predicted")
print("-" * (12 + 12 * len(labels)))

for i, label in enumerate(labels):
    print(f"{label:>12}", end="")
    for j in range(len(labels)):
        print(f"{cm[i,j]:>12,}", end="")
    print(f"  | {label}")

print("\n↑ Actual")

# Calculate per-class accuracy
print("\n📊 Per-class accuracy:")
for i, label in enumerate(labels):
    class_total = cm[i, :].sum()
    class_correct = cm[i, i]
    class_acc = class_correct / class_total if class_total > 0 else 0

    status = "✅" if class_acc > 0.6 else "⚠️" if class_acc > 0.4 else "❌"
    print(f"   {status} {label:>10}: {class_acc:.2%} ({class_correct:,}/{class_total:,})")



📊 CONFUSION MATRIX (TEST SET)

                    high         low      medium   very_high ← Predicted
------------------------------------------------------------
        high      77,396      13,027      49,674      59,295  | high
         low      50,931     710,050     203,943      21,259  | low
      medium      90,394     111,997     222,594      40,380  | medium
   very_high       9,858         800       2,603      44,717  | very_high

↑ Actual

📊 Per-class accuracy:
   ❌       high: 38.82% (77,396/199,392)
   ✅        low: 72.00% (710,050/986,183)
   ⚠️     medium: 47.83% (222,594/465,365)
   ✅  very_high: 77.13% (44,717/57,978)


In [15]:
# ============================================
# CELL 9: FEATURE IMPORTANCE
# ============================================

print("\n" + "=" * 70)
print("🔝 FEATURE IMPORTANCE (XGBOOST)")
print("=" * 70)

importances = model.feature_importances_
feature_names = X.columns.tolist()
indices = np.argsort(importances)[::-1]

print(f"\nTop 15 most important features:")
print("-" * 70)
for i in range(min(15, len(feature_names))):
    idx = indices[i]
    bar = "█" * int(importances[idx] * 50)
    print(f"{i+1:2d}. {feature_names[idx]:<25} {importances[idx]:.4f} {bar}")

# Cumulative importance
print(f"\n📊 Cumulative importance:")
cumsum = 0
for i in range(len(importances)):
    cumsum += importances[indices[i]]
    if cumsum >= 0.5 and i < 10:
        print(f"   Top {i+1} features explain 50% of importance")
        break

cumsum = 0
for i in range(len(importances)):
    cumsum += importances[indices[i]]
    if cumsum >= 0.8:
        print(f"   Top {i+1} features explain 80% of importance")
        break


🔝 FEATURE IMPORTANCE (XGBOOST)

Top 15 most important features:
----------------------------------------------------------------------
 1. is_rush_hour              0.2535 ████████████
 2. hour                      0.1334 ██████
 3. time_of_day               0.0905 ████
 4. is_weekend                0.0731 ███
 5. direction_id              0.0708 ███
 6. trip_number               0.0643 ███
 7. trip_stage                0.0586 ██
 8. day_of_week               0.0544 ██
 9. route_short_name          0.0477 ██
10. pt_sequence               0.0474 ██
11. register_code             0.0415 ██
12. route_percentage          0.0353 █
13. end_trip                  0.0296 █
14. month                     0.0000 

📊 Cumulative importance:
   Top 4 features explain 50% of importance
   Top 9 features explain 80% of importance


In [16]:
# ============================================
# CELL 10: COMPARE WITH BASELINE AND RF
# ============================================
print("\n" + "=" * 70)
print("📈 COMPARISON WITH BASELINE AND RANDOM FOREST")
print("=" * 70)

# Baseline: always predict majority class
majority_class_label = y_train_original.value_counts().idxmax()
baseline_acc = (y_test_original == majority_class_label).mean()

print(f"\n1️⃣  Baseline (always predict '{majority_class_label}'):")
print(f"      Accuracy: {baseline_acc:.4f}")

print(f"\n2️⃣  Random Forest (your previous results):")
print(f"      Accuracy: 0.5920")
print(f"      Balanced Accuracy: 0.5677")
print(f"      F1 Macro: 0.4753")

print(f"\n3️⃣  XGBoost (current model):")
print(f"      Accuracy: {acc_test:.4f}")
print(f"      Balanced Accuracy: {bal_acc_test:.4f}")
print(f"      F1 Macro: {f1_macro_test:.4f}")

# Calculate improvements
rf_bal_acc = 0.5677
rf_f1_macro = 0.4753

bal_acc_improvement = (bal_acc_test - rf_bal_acc) / rf_bal_acc * 100
f1_improvement = (f1_macro_test - rf_f1_macro) / rf_f1_macro * 100

print(f"\n📊 XGBoost vs Random Forest:")
print(f"   Balanced Accuracy: {bal_acc_improvement:+.2f}% improvement")
print(f"   F1 Macro: {f1_improvement:+.2f}% improvement")

if bal_acc_test > rf_bal_acc:
    print(f"\n✅ XGBoost performs BETTER than Random Forest")
elif bal_acc_test > rf_bal_acc - 0.01:
    print(f"\n⚠️  XGBoost performs SIMILAR to Random Forest")
else:
    print(f"\n❌ Random Forest performs BETTER (unexpected)")




📈 COMPARISON WITH BASELINE AND RANDOM FOREST

1️⃣  Baseline (always predict 'low'):
      Accuracy: 0.5771

2️⃣  Random Forest (your previous results):
      Accuracy: 0.5920
      Balanced Accuracy: 0.5677
      F1 Macro: 0.4753

3️⃣  XGBoost (current model):
      Accuracy: 0.6172
      Balanced Accuracy: 0.5894
      F1 Macro: 0.5031

📊 XGBoost vs Random Forest:
   Balanced Accuracy: +3.83% improvement
   F1 Macro: +5.85% improvement

✅ XGBoost performs BETTER than Random Forest


In [17]:
# ============================================
# CELL 11: FINAL SUMMARY
# ============================================
print("\n" + "=" * 70)
print("🎉 FINAL SUMMARY")
print("=" * 70)

print(f"""
📊 DATASET:
   • Total records: {len(X):,}
   • Features: {len(X.columns)}
   • Classes: {len(labels)} ({', '.join(labels)})
   • Imbalance ratio: {imbalance_ratio:.1f}:1

🤖 MODEL:
   • Algorithm: XGBoost
   • Using: sample_weight ✅
   • n_estimators: {model.n_estimators}
   • max_depth: {model.max_depth}
   • learning_rate: {model.learning_rate}

📈 PERFORMANCE:
   • Accuracy: {acc_test:.4f}
   • Balanced Accuracy: {bal_acc_test:.4f} ⭐
   • F1 Macro: {f1_macro_test:.4f} ⭐
   • Baseline: {baseline_acc:.4f}
   • Overfitting gap: {gap:.4f}

💡 INTERPRETATION:
   • Balanced Accuracy of {bal_acc_test:.2%} means the model
     correctly classifies ~{bal_acc_test*100:.0f}% across ALL classes
   • This is {'GOOD' if bal_acc_test > 0.6 else 'ACCEPTABLE' if bal_acc_test > 0.5 else 'NEEDS IMPROVEMENT'} for imbalanced data

🎯 NEXT STEPS TO IMPROVE:
   1. Add smart features (route × hour interactions)
   2. Adjust thresholds (25→30%, 50→55%)
   3. Tune hyperparameters (grid search)
   4. Consider cost-sensitive learning
""")

print("=" * 70)
print("✅ Training complete!")
print("=" * 70)


🎉 FINAL SUMMARY

📊 DATASET:
   • Total records: 8,544,587
   • Features: 14
   • Classes: 4 (high, low, medium, very_high)
   • Imbalance ratio: 17.0:1

🤖 MODEL:
   • Algorithm: XGBoost
   • Using: sample_weight ✅
   • n_estimators: 300
   • max_depth: 8
   • learning_rate: 0.05

📈 PERFORMANCE:
   • Accuracy: 0.6172
   • Balanced Accuracy: 0.5894 ⭐
   • F1 Macro: 0.5031 ⭐
   • Baseline: 0.5771
   • Overfitting gap: 0.0039

💡 INTERPRETATION:
   • Balanced Accuracy of 58.94% means the model
     correctly classifies ~59% across ALL classes
   • This is ACCEPTABLE for imbalanced data

🎯 NEXT STEPS TO IMPROVE:
   1. Add smart features (route × hour interactions)
   2. Adjust thresholds (25→30%, 50→55%)
   3. Tune hyperparameters (grid search)
   4. Consider cost-sensitive learning

✅ Training complete!


In [18]:
# ============================================
# CELL 12: SAVE MODEL
# ============================================
model_filename = 'sunt_xgboost_model_weighted.pkl'
with open(model_filename, 'wb') as f:
    pickle.dump(model, f)
print(f"\n✅ Model saved to {model_filename}")

# Save feature names for later use
feature_names_file = 'feature_names.pkl'
with open(feature_names_file, 'wb') as f:
    pickle.dump(feature_names, f)
print(f"✅ Feature names saved to {feature_names_file}")

print("\n📁 Files ready for deployment:")
print(f"   • {model_filename}")
print(f"   • {feature_names_file}")


✅ Model saved to sunt_xgboost_model_weighted.pkl
✅ Feature names saved to feature_names.pkl

📁 Files ready for deployment:
   • sunt_xgboost_model_weighted.pkl
   • feature_names.pkl
